# 🌿 Leaf disease model training

**Before you start**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. Your Google Drive folder `agri_ml` must contain: `kaggle.json`, `train_disease.py`, `merge_disease_datasets.py`, `plantdoc.json`
3. You must have clicked **Join Competition** on Kaggle's *paddy-disease-classification* page.

Then click **Runtime → Run all**. When Google asks for Drive permission, click **Allow**.
If Colab disconnects, just **Run all** again — finished steps are skipped.

## 1. Settings (you can change these)

In [ ]:
DRIVE = '/content/drive/MyDrive/agri_ml'   # your Drive folder
OUT = DRIVE + '/model_v2'                  # where the new model is saved (old models are not touched)

USE_PLANTVILLAGE = True    # extra lab photos: helps, but only a little per class
MAX_PER_CLASS = 500        # max photos per disease, so lab photos do not drown out field photos
RICE_TEST_FRACTION = 0.10  # share of rice photos kept aside to test rice honestly

ARCH, IMG_SIZE, BATCH = 'efficientnetb3', 300, 16   # strongest; about 1.5-2.5 hours
# ARCH, IMG_SIZE, BATCH = 'efficientnetb0', 224, 32 # faster (about 40 min), a bit less accurate
EPOCHS, FINE_TUNE_EPOCHS = 8, 15

## 2. Check the GPU

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu or 'NO GPU!')
assert 'GPU' in gpu, 'No GPU. Go to Runtime -> Change runtime type -> T4 GPU, then Run all again.'

## 3. Connect Google Drive and check your files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
need = ['kaggle.json', 'train_disease.py', 'merge_disease_datasets.py', 'plantdoc.json']
missing = [f for f in need if not os.path.exists(f'{DRIVE}/{f}')]
assert not missing, f'Upload these to Google Drive folder agri_ml first: {missing}'
assert 'accuracy_crop_known' in open(f'{DRIVE}/train_disease.py').read(), \
    'Your train_disease.py is old. Upload the newest one from agri-chatBot/ml/ to Drive (replace it).'
print('All files found ✅')

## 4. Kaggle login

In [ ]:
!mkdir -p ~/.kaggle && cp "{DRIVE}/kaggle.json" ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
print('Kaggle ready ✅')

## 5. Download datasets (skipped if already downloaded)

In [ ]:
%cd /content
import glob
if not os.path.isdir('raw/paddy/train_images'):
    !kaggle competitions download -c paddy-disease-classification -p raw/paddy
    !unzip -q -o raw/paddy/paddy-disease-classification.zip -d raw/paddy
assert os.path.isdir('raw/paddy/train_images'), 'Rice download failed. Did you click Join Competition on Kaggle?'

if not os.path.isdir('raw/plantdoc/train'):
    !git clone -q https://github.com/pratikkayal/PlantDoc-Dataset raw/plantdoc

PV = ''
if USE_PLANTVILLAGE:
    if not glob.glob('raw/plantvillage/**/color', recursive=True):
        !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p raw/plantvillage --unzip
    found = glob.glob('/content/raw/plantvillage/**/color', recursive=True)
    PV = found[0] if found else ''
    print('PlantVillage:', PV or 'NOT FOUND - continuing without it')
print('Downloads ready ✅')

### (Optional) Add more datasets
On a Kaggle dataset page click **⋮ → Copy API command**, then add a line below like:
`!kaggle datasets download -d OWNER/NAME -p /content/raw/tomato2 --unzip`
and add its folder to `EXTRA` as `'/content/raw/tomato2/FOLDER:Tomato'` (`:Crop` only if its folders lack the crop name).
**Only pick datasets whose photos have real field backgrounds.**

In [ ]:
EXTRA = []   # e.g. ['/content/raw/tomato2/train:Tomato']

## 6. Merge all photos into one folder

In [ ]:
!rm -rf /content/merged /content/field_test
sources = ['/content/raw/plantdoc/train', '/content/raw/paddy/train_images:Rice'] + EXTRA + ([PV] if PV else [])
args = ' '.join(f'"{s}"' for s in sources)
!python "{DRIVE}/merge_disease_datasets.py" /content/merged {args} --map "{DRIVE}/plantdoc.json" --max-per-class {MAX_PER_CLASS}

## 7. Make the field test set (photos the model will never train on)

In [ ]:
import random, shutil
!python "{DRIVE}/merge_disease_datasets.py" /content/field_test /content/raw/plantdoc/test --map "{DRIVE}/plantdoc.json" --min-images 1

# Move a share of every rice class out of training into the test set
random.seed(42)
moved = 0
for cls in sorted(os.listdir('/content/merged')):
    if not cls.startswith('Rice___'):
        continue
    files = sorted(os.listdir(f'/content/merged/{cls}'))
    pick = random.sample(files, max(1, int(len(files) * RICE_TEST_FRACTION)))
    os.makedirs(f'/content/field_test/{cls}', exist_ok=True)
    for f in pick:
        shutil.move(f'/content/merged/{cls}/{f}', f'/content/field_test/{cls}/{f}')
    moved += len(pick)
print(f'Training classes: {len(os.listdir("/content/merged"))}')
print(f'Field test: {len(os.listdir("/content/field_test"))} classes, rice photos held out: {moved} ✅')

## 8. Train (the long step — keep this tab open)

In [ ]:
!python "{DRIVE}/train_disease.py" --data /content/merged --test-dir /content/field_test --out "{OUT}" \
    --arch {ARCH} --img-size {IMG_SIZE} --batch-size {BATCH} --epochs {EPOCHS} --fine-tune-epochs {FINE_TUNE_EPOCHS}

## 9. Results

In [ ]:
import json
m = json.load(open(f'{OUT}/disease_metrics.json'))
t = m.get('test', {})
print(f"Model: {m['arch']} @ {m['img_size']}px, {m['num_classes']} classes")
print(f"Lab-style validation accuracy: {m['val_accuracy']:.1%}")
if t.get('n'):
    print(f"FIELD accuracy (crop known, like the app): {t['accuracy_crop_known']:.1%}   <- the number that matters")
    print(f"FIELD accuracy (crop not given):           {t['accuracy']:.1%}")
    print('\nPer disease (field, crop known):')
    for name, v in sorted(t['per_class'].items(), key=lambda kv: kv[1]['accuracy_crop_known']):
        print(f"  {v['accuracy_crop_known']:6.0%}  {name}  ({v['n']} photos)")
print(f'\nDownload these 3 files from Google Drive folder {OUT}:')
print('  disease_cnn.keras, disease_classes.json, disease_metrics.json')